# CVPO Level 0: Image Classification (True Hello World)

## INTRO
This notebook walks you through the simplest computer vision task: **classification**.
You give the system one image and a set of candidate labels. It tells you which label best describes the image.

## EXPECTATIONS
By the end of this notebook you will:
- Understand what zero-shot classification means
- Run a real CLIP model through CVPO's pipeline on your own image
- See how changing labels affects confidence scores
- Learn why this is the foundation everything else builds on

In [ ]:
# CONFIGURATION
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT / "src"))

IMAGE_PATH = ROOT / "libs" / "capybara.jpg"  # Change to your image
LABELS = ["capybara", "beaver", "otter", "bear", "dog"]  # Change to your labels
BACKEND = "siglip"  # "deterministic" for no-download test, "siglip" for real CLIP model

## WARMUP — See what the pipeline does before building it yourself

Here we run the Level 0 pipeline on a sample image to see the output format.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from cvpo.pipelines import build_level0_pipeline
from cvpo.core.data_types import PipelineContext

image = np.array(Image.open(IMAGE_PATH).convert("RGB"), dtype=np.uint8)
plt.imshow(image)
plt.title(f"Input: {IMAGE_PATH.name} ({image.shape[1]}x{image.shape[0]})")
plt.axis("off")
plt.show()
print(f"Image shape: {image.shape}, dtype: {image.dtype}")
print(f"Labels: {LABELS}")

## BUILD — Construct and run the classification pipeline

In [ ]:
pipeline = build_level0_pipeline(candidate_labels=LABELS, backend=BACKEND)
ctx = PipelineContext(run_id="notebook-l0", input_source=str(IMAGE_PATH), frontend="notebook")

result = pipeline.run(image, ctx)

print(f"Top label: {result.classes[0].label}")
print(f"Confidence: {result.classes[0].confidence:.4f}")
print(f"\nAll scores:")
for label, score in sorted(result.classes[0].scores, key=lambda x: -x[1]):
    bar = "█" * int(score * 50)
    print(f"  {label:20s} {score:.4f} {bar}")

## MEASURE — What happens when you change the labels?

Try removing the correct label and see how the model picks the next-closest option.
This demonstrates *contrastive scoring* — the model only picks from what you give it.

In [ ]:
labels_without_correct = [l for l in LABELS if l != "capybara"]
pipeline_shifted = build_level0_pipeline(candidate_labels=labels_without_correct, backend=BACKEND)
result_shifted = pipeline_shifted.run(image, ctx)

print(f"Labels WITHOUT 'capybara': {labels_without_correct}")
print(f"Top label now: {result_shifted.classes[0].label}")
print(f"Confidence: {result_shifted.classes[0].confidence:.4f}")
print(f"\nAll scores:")
for label, score in sorted(result_shifted.classes[0].scores, key=lambda x: -x[1]):
    bar = "█" * int(score * 50)
    print(f"  {label:20s} {score:.4f} {bar}")
print("\nNotice: the model picked the closest available option, not 'capybara'.")

## ANALYSIS — What did we learn?

**Key concepts from this notebook:**

1. **Classification** answers "what is this?" for the whole image — one input, one answer.
2. **Zero-shot** means the model was never specifically trained on your labels. It learned a general understanding of images and text from hundreds of millions of examples, and transfers that to whatever labels you provide.
3. **Contrastive scoring** means the model scores each label *relative* to the others. Removing or adding labels changes every score. The confidence number is a relative preference, not an absolute probability — technically called a softmax distribution over the candidate set.
4. **Label quality matters**: specific, non-overlapping labels produce clearer results.

**What this does NOT do:**
- It doesn't tell you WHERE objects are (that's detection — Level 1)
- It doesn't draw boundaries around objects (that's segmentation)
- It doesn't track anything over time (that's tracking — Level 3)

## CONFIRMATION
You have successfully run a real vision-language model (CLIP) through CVPO's typed pipeline abstraction. The next level (Level 1) adds spatial localization by chaining a detection model before classification.